Sequential Chats and Customer Onboarding

## Setup

In [5]:
import sys
print("Python:", sys.executable)

import autogen
print("AutoGen:", autogen.__version__)

import google.genai
print("google-genai: OK")

Python: C:\Users\gnana\anaconda3\envs\ai_project\python.exe
AutoGen: 0.7.2
google-genai: OK


In [4]:
#from utils import get_openai_api_key
#OPENAI_API_KEY = get_openai_api_key()
##print(OPENAI_API_KEY)
#llm_config = {"model": "gpt-3.5-turbo","api_key":OPENAI_API_KEY}

In [6]:
from utils import get_gemini_api_key

GEMINI_API_KEY = get_gemini_api_key()

print("Gemini API key loaded:", bool(GEMINI_API_KEY))

Gemini API key loaded: True


In [7]:
llm_config = {
    "config_list": [
        {
            "model": "gemini-3.6-flash",
            "api_key": GEMINI_API_KEY,
            "api_type": "google"
        }
    ],
    "temperature": 0.2,
}

In [8]:
from autogen import ConversableAgent

test_agent = ConversableAgent(
    name="test_agent",
    system_message="You are a helpful assistant.",
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)

customer_proxy_agent = ConversableAgent(
    name="customer_proxy_agent",
    llm_config=False,
    code_execution_config=False,
    human_input_mode="NEVER",
)

In [9]:
result = customer_proxy_agent.initiate_chat(
    test_agent,
    message="Say hello and tell me your name.",
    max_turns=1,
)

print(result)

customer_proxy_agent (to test_agent):

Say hello and tell me your name.

--------------------------------------------------------------------------------
test_agent (to customer_proxy_agent):

Hello! I'm Gemini, a helpful AI assistant. How can I help you today?

--------------------------------------------------------------------------------
ChatResult(chat_id=None, chat_history=[{'content': 'Say hello and tell me your name.', 'role': 'assistant', 'name': 'customer_proxy_agent'}, {'content': "Hello! I'm Gemini, a helpful AI assistant. How can I help you today?", 'role': 'user', 'name': 'test_agent'}], summary="Hello! I'm Gemini, a helpful AI assistant. How can I help you today?", cost={'usage_including_cached_inference': {'total_cost': 0, 'gemini-3.6-flash': {'cost': 0, 'prompt_tokens': 16, 'completion_tokens': 19, 'total_tokens': 35}}, 'usage_excluding_cached_inference': {'total_cost': 0}}, human_input=[])


## Creating the needed agents

In [10]:
from utils import get_gemini_api_key
from autogen import ConversableAgent

GEMINI_API_KEY = get_gemini_api_key()

llm_config = {
    "config_list": [
        {
            "model": "gemini-3.6-flash",
            "api_key": GEMINI_API_KEY,
            "api_type": "google"
        }
    ],
    "temperature": 0.2,
}

print("Gemini API key loaded:", bool(GEMINI_API_KEY))
print("LLM configuration ready")

Gemini API key loaded: True
LLM configuration ready


In [11]:
onboarding_personal_information_agent = ConversableAgent(
    name="onboarding_personal_information_agent",

    system_message="""
You are a helpful customer onboarding agent.

You are here to help new customers get started with our product.

Your job is to gather the customer's name and location.

Do not ask for other information.

Return 'TERMINATE' when you have gathered all the information.
""",

    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)


onboarding_topic_preference_agent = ConversableAgent(
    name="onboarding_topic_preference_agent",

    system_message="""
You are a helpful customer onboarding agent.

You are here to help new customers get started with our product.

Your job is to gather the customer's preferences on news topics.

Do not ask for other information.

Return 'TERMINATE' when you have gathered all the information.
""",

    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)


customer_engagement_agent = ConversableAgent(
    name="customer_engagement_agent",

    system_message="""
You are a helpful customer service agent.

You are here to provide fun content for the customer based on
their personal information and topic preferences.

This could include fun facts, jokes, or interesting stories.

Make sure to make it engaging and fun!

Return 'TERMINATE' when you are done.
""",

    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",

    is_termination_msg=lambda msg:
        "terminate" in (msg.get("content") or "").lower(),
)


customer_proxy_agent = ConversableAgent(
    name="customer_proxy_agent",

    llm_config=False,
    code_execution_config=False,
    human_input_mode="ALWAYS",

    is_termination_msg=lambda msg:
        "terminate" in (msg.get("content") or "").lower(),
)

print("All 4 agents created successfully!")

All 4 agents created successfully!


## Creating tasks

Now, you can craft a series of tasks to facilitate the onboarding process.

In [12]:
chats = [
    {
        "sender": onboarding_personal_information_agent,
        "recipient": customer_proxy_agent,
        "message": "Hello, I'm here to help you get started with our product. Could you tell me your name and location?",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Return the customer information as JSON object only: {'name': '', 'location': ''}",
        },
        "max_turns": 1,
        "clear_history": True
    },
    {
        "sender": onboarding_topic_preference_agent,
        "recipient": customer_proxy_agent,
        "message": "Great! Could you tell me what topics you are interested in reading about?",
        "summary_method": "reflection_with_llm",
        "max_turns": 1,
        "clear_history": False
    },
    {
        "sender": customer_proxy_agent,
        "recipient": customer_engagement_agent,
        "message": "Let's find something fun to read.",
        "max_turns": 1,
        "summary_method": "reflection_with_llm",
    },
]

Start the onboarding process

In [14]:
chat_results = []

for chat in chats:

    kwargs = {
        "message": chat["message"],
        "max_turns": chat.get("max_turns", 1),
        "clear_history": chat.get("clear_history", False),
        "summary_method": chat.get("summary_method"),
    }

    # Only add summary_args if it exists
    if chat.get("summary_args") is not None:
        kwargs["summary_args"] = chat["summary_args"]

    result = chat["sender"].initiate_chat(
        chat["recipient"],
        **kwargs
    )

    chat_results.append(result)

print("All chats completed!")

onboarding_personal_information_agent (to customer_proxy_agent):

Hello, I'm here to help you get started with our product. Could you tell me your name and location?

--------------------------------------------------------------------------------


Replying as customer_proxy_agent. Provide feedback to onboarding_personal_information_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  i am uma from salem


customer_proxy_agent (to onboarding_personal_information_agent):

i am uma from salem

--------------------------------------------------------------------------------
onboarding_topic_preference_agent (to customer_proxy_agent):

Great! Could you tell me what topics you are interested in reading about?

--------------------------------------------------------------------------------


Replying as customer_proxy_agent. Provide feedback to onboarding_topic_preference_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  I am interested in Artificial Intelligence, Data Science, Python and technology.


customer_proxy_agent (to onboarding_topic_preference_agent):

I am interested in Artificial Intelligence, Data Science, Python and technology.

--------------------------------------------------------------------------------
customer_proxy_agent (to customer_engagement_agent):

Let's find something fun to read.

--------------------------------------------------------------------------------
customer_engagement_agent (to customer_proxy_agent):

Hello there! Welcome! I am super excited to be your personal fun-content guide today! 🌟

To help me tailor the absolute best facts, mind-blowing stories, or hilarious jokes just for you, could you share a little bit about what you like? 

For example:
* **What are your favorite topics?** (e.g., Space, Animals, History, Tech, Pop Culture, Food, Science)
* **What kind of vibe are you looking for today?** (e.g., Quick funny jokes, cool mind-bending facts, or a fascinating short story?)
* **Any favorite hobbies or fun details about yourself you'd li

**Note**: You might get a slightly different response than what's shown in the video. Feel free to try different inputs, such as name, location, and preferences.

## Print out the summary

In [20]:
for i, result in enumerate(chat_results, 1):
    print(f"\n========== CHAT {i} ==========")
    print(result.chat_history)


========== CHAT 1 ==========
[{'content': "Hello, I'm here to help you get started with our product. Could you tell me your name and location?", 'role': 'assistant', 'name': 'onboarding_personal_information_agent'}, {'content': 'i am uma from salem', 'role': 'user', 'name': 'customer_proxy_agent'}]

========== CHAT 2 ==========
[{'content': 'Great! Could you tell me what topics you are interested in reading about?', 'role': 'assistant', 'name': 'onboarding_topic_preference_agent'}, {'content': 'I am interested in Artificial Intelligence, Data Science, Python and technology.', 'role': 'user', 'name': 'customer_proxy_agent'}, {'content': 'Great! Could you tell me what topics you are interested in reading about?', 'role': 'assistant', 'name': 'onboarding_topic_preference_agent'}, {'content': 'I am interested in Artificial Intelligence, Data Science, Python and technology.', 'role': 'user', 'name': 'customer_proxy_agent'}]

========== CHAT 3 ==========
[{'content': "Let's find something f

In [24]:
import pandas as pd
import json

# Get the content from summary[0]
data_str = summary[0]["content"]

# Convert JSON string to dictionary
data_dict = json.loads(data_str)

# Convert dictionary to DataFrame
df = pd.DataFrame([data_dict])

df

,name,location
0,Uma,Salem


In [18]:
!pip install pandas

In [23]:
import pandas as pd
import ast
import json

data_str = summary[0]["content"]

data_dict = json.loads(data_str)

df = pd.DataFrame([data_dict])

df

,name,location
0,Uma,Salem


In [25]:
df["TopicPreference"] = summary[1]["content"]

In [26]:
df

,name,location,TopicPreference
0,Uma,Salem,The primary topics of interest are Artificial ...


In [28]:
df["Engagement"] = summary[2]["content"]

In [29]:
df

,name,location,TopicPreference,Enagement,Engagement
0,Uma,Salem,The primary topics of interest are Artificial ...,NaN,"The user is looking for fun reading material, ..."


In [30]:
!pip install gspread
!pip install oauth2client


   -------------------- ------------------- 1/2 [gspread]
   ---------------------------------------- 2/2 [gspread]


   ---------------------------------------- 0/2 [rsa]
   -------------------- ------------------- 1/2 [oauth2client]
   -------------------- ------------------- 1/2 [oauth2client]
   ---------------------------------------- 2/2 [oauth2client]



In [33]:
def add_leads_to_google_sheet(df):
    import gspread
    from oauth2client.service_account import ServiceAccountCredentials

    # Google Sheets API scope
    scope = [
        "https://www.googleapis.com/auth/spreadsheets"
    ]

    # Load Google service account credentials
    creds = ServiceAccountCredentials.from_json_keyfile_name(
        "websiteleadchatgptapi-a46f50ae1507.json",
        scope
    )

    # Authorize Google Sheets
    client = gspread.authorize(creds)

    # Google Sheet ID
    sheet_id = "1ZSzaYLeD9bsaY6-ZSH0Yna6OqRe_Pd7-j1nFcCBdLQE"

    # Open spreadsheet
    spreadsheet = client.open_by_key(sheet_id)

    # Select first worksheet
    worksheet = spreadsheet.get_worksheet(0)

    # Add each DataFrame row
    for _, row in df.iterrows():
        worksheet.append_row(row.tolist())

    print("Data added successfully!")

    return "added"

In [36]:
#add_leads_to_google_sheet(df)

## Print out the cost

In [35]:
for chat_result in chat_results:
    print(chat_result.cost)
    print("\n")

{'usage_including_cached_inference': {'total_cost': 0, 'gemini-3.6-flash': {'cost': 0, 'prompt_tokens': 103, 'completion_tokens': 24, 'total_tokens': 127}}, 'usage_excluding_cached_inference': {'total_cost': 0, 'gemini-3.6-flash': {'cost': 0, 'prompt_tokens': 103, 'completion_tokens': 24, 'total_tokens': 127}}}


{'usage_including_cached_inference': {'total_cost': 0, 'gemini-3.6-flash': {'cost': 0, 'prompt_tokens': 78, 'completion_tokens': 17, 'total_tokens': 95}}, 'usage_excluding_cached_inference': {'total_cost': 0, 'gemini-3.6-flash': {'cost': 0, 'prompt_tokens': 78, 'completion_tokens': 17, 'total_tokens': 95}}}


{'usage_including_cached_inference': {'total_cost': 0, 'gemini-3.6-flash': {'cost': 0, 'prompt_tokens': 271, 'completion_tokens': 203, 'total_tokens': 474}}, 'usage_excluding_cached_inference': {'total_cost': 0, 'gemini-3.6-flash': {'cost': 0, 'prompt_tokens': 271, 'completion_tokens': 203, 'total_tokens': 474}}}


